In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(df.head())
print()
print(df.columns.tolist())
print()
print(df.shape)

    part_id      category  unit_price  stock_quantity  min_stock_level  \
0  PART_001    Suspension     4583.30              10               21   
1  PART_002  Transmission     4265.19              91               18   
2  PART_003    Electrical     2302.31              56               25   
3  PART_004  Transmission      567.51              88                7   
4  PART_005  Transmission     1917.01              49                5   

   max_stock_level  monthly_usage  reserved_quantity  lead_time_days  \
0               50             36                 12              10   
1              107             25                  3              10   
2               50             21                  3              19   
3               83             36                  5              17   
4              145             10                 18              21   

   weekly_usage  lead_time_demand  
0          9.00         12.857143  
1          6.25          8.928571  
2          5.2

In [27]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

['part_id', 'category', 'unit_price', 'stock_quantity', 'min_stock_level', 'max_stock_level', 'monthly_usage', 'reserved_quantity', 'lead_time_days', 'weekly_usage', 'lead_time_demand']


In [28]:
df = df.copy()

numeric_columns = [
    "unit_price",
    "stock_quantity",
    "min_stock_level",
    "max_stock_level",
    "monthly_usage",
    "reserved_quantity",
    "lead_time_days"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df[numeric_columns] = df[numeric_columns].fillna(0)

print(df.isnull().sum())

part_id              0
category             0
unit_price           0
stock_quantity       0
min_stock_level      0
max_stock_level      0
monthly_usage        0
reserved_quantity    0
lead_time_days       0
weekly_usage         0
lead_time_demand     0
dtype: int64


In [31]:
df["available_stock"] = (
    df["stock_quantity"]
    - df["reserved_quantity"]
).clip(lower=0)
df[
    [
        "part_id",
        "stock_quantity",
        "reserved_quantity",
        "available_stock"
    ]
].head(10)

,part_id,stock_quantity,reserved_quantity,available_stock
0,PART_001,10,12,0
1,PART_002,91,3,88
2,PART_003,56,3,53
3,PART_004,88,5,83
4,PART_005,49,18,31
5,PART_006,22,11,11
6,PART_007,30,6,24
7,PART_008,93,9,84
8,PART_009,41,18,23
9,PART_010,98,6,92


In [32]:
df["weekly_usage"] = (
    df["monthly_usage"] / 4
)
df[
    [
        "part_id",
        "monthly_usage",
        "weekly_usage"
    ]
].head(10)

,part_id,monthly_usage,weekly_usage
0,PART_001,36,9.00
1,PART_002,25,6.25
2,PART_003,21,5.25
3,PART_004,36,9.00
4,PART_005,10,2.50
5,PART_006,37,9.25
6,PART_007,9,2.25
7,PART_008,24,6.00
8,PART_009,35,8.75
9,PART_010,49,12.25


In [33]:
df["lead_time_demand"] = (
    (df["weekly_usage"] / 7)
    * df["lead_time_days"]
)

df[
    [
        "part_id",
        "weekly_usage",
        "lead_time_days",
        "lead_time_demand"
    ]
].head(10)

,part_id,weekly_usage,lead_time_days,lead_time_demand
0,PART_001,9.00,10,12.857143
1,PART_002,6.25,10,8.928571
2,PART_003,5.25,19,14.250000
3,PART_004,9.00,17,21.857143
4,PART_005,2.50,21,7.500000
5,PART_006,9.25,14,18.500000
6,PART_007,2.25,9,2.892857
7,PART_008,6.00,14,12.000000
8,PART_009,8.75,1,1.250000
9,PART_010,12.25,19,33.250000


In [34]:
df["reorder_required"] = (
    (df["available_stock"] <= df["min_stock_level"])
    |
    (df["available_stock"] < df["lead_time_demand"])
)
df[
    [
        "part_id",
        "available_stock",
        "min_stock_level",
        "lead_time_demand",
        "reorder_required"
    ]
].head(20)

,part_id,available_stock,min_stock_level,lead_time_demand,reorder_required
0,PART_001,0,21,12.857143,True
1,PART_002,88,18,8.928571,False
2,PART_003,53,25,14.250000,False
3,PART_004,83,7,21.857143,False
4,PART_005,31,5,7.500000,False
5,PART_006,11,24,18.500000,True
6,PART_007,24,25,2.892857,True
7,PART_008,84,27,12.000000,False
8,PART_009,23,5,1.250000,False
9,PART_010,92,7,33.250000,False


In [35]:
df["reorder_quantity"] = (
    df["max_stock_level"]
    - df["available_stock"]
).clip(lower=0)
df.loc[
    ~df["reorder_required"],
    "reorder_quantity"
] = 0
df[
    [
        "part_id",
        "available_stock",
        "max_stock_level",
        "reorder_required",
        "reorder_quantity"
    ]
].head(20)

,part_id,available_stock,max_stock_level,reorder_required,reorder_quantity
0,PART_001,0,50,True,50
1,PART_002,88,107,False,0
2,PART_003,53,50,False,0
3,PART_004,83,83,False,0
4,PART_005,31,145,False,0
5,PART_006,11,97,True,86
6,PART_007,24,138,True,114
7,PART_008,84,50,False,0
8,PART_009,23,65,False,0
9,PART_010,92,110,False,0


In [36]:
df["estimated_reorder_cost"] = (
    df["reorder_quantity"]
    * df["unit_price"]
)
df[
    [
        "part_id",
        "reorder_quantity",
        "unit_price",
        "estimated_reorder_cost"
    ]
].head(10)

,part_id,reorder_quantity,unit_price,estimated_reorder_cost
0,PART_001,50,4583.30,229165.00
1,PART_002,0,4265.19,0.00
2,PART_003,0,2302.31,0.00
3,PART_004,0,567.51,0.00
4,PART_005,0,1917.01,0.00
5,PART_006,86,3377.32,290449.52
6,PART_007,114,3363.02,383384.28
7,PART_008,0,2997.36,0.00
8,PART_009,0,1446.14,0.00
9,PART_010,0,2850.09,0.00


In [38]:
df["reorder_reason"] = "No reorder needed"

df.loc[
    df["available_stock"] <= df["min_stock_level"],
    "reorder_reason"
] = "Stock is at or below minimum level"

df.loc[
    (df["available_stock"] > df["min_stock_level"])
    &
    (df["available_stock"] < df["lead_time_demand"]),
    "reorder_reason"
] = "Stock may not cover lead-time demand"
df[
    [
        "part_id",
        "available_stock",
        "min_stock_level",
        "lead_time_demand",
        "reorder_required",
        "reorder_reason"
    ]
].head(20)

,part_id,available_stock,min_stock_level,lead_time_demand,reorder_required,reorder_reason
0,PART_001,0,21,12.857143,True,Stock is at or below minimum level
1,PART_002,88,18,8.928571,False,No reorder needed
2,PART_003,53,25,14.250000,False,No reorder needed
3,PART_004,83,7,21.857143,False,No reorder needed
4,PART_005,31,5,7.500000,False,No reorder needed
5,PART_006,11,24,18.500000,True,Stock is at or below minimum level
6,PART_007,24,25,2.892857,True,Stock is at or below minimum level
7,PART_008,84,27,12.000000,False,No reorder needed
8,PART_009,23,5,1.250000,False,No reorder needed
9,PART_010,92,7,33.250000,False,No reorder needed


In [39]:
df["urgency"] = "LOW"
df.loc[
    df["available_stock"] <= df["min_stock_level"],
    "urgency"
] = "MEDIUM"
df.loc[
    df["available_stock"] < df["lead_time_demand"],
    "urgency"
] = "HIGH"
df[
    [
        "part_id",
        "available_stock",
        "min_stock_level",
        "lead_time_demand",
        "reorder_required",
        "urgency"
    ]
].head(20)

,part_id,available_stock,min_stock_level,lead_time_demand,reorder_required,urgency
0,PART_001,0,21,12.857143,True,HIGH
1,PART_002,88,18,8.928571,False,LOW
2,PART_003,53,25,14.250000,False,LOW
3,PART_004,83,7,21.857143,False,LOW
4,PART_005,31,5,7.500000,False,LOW
5,PART_006,11,24,18.500000,True,HIGH
6,PART_007,24,25,2.892857,True,MEDIUM
7,PART_008,84,27,12.000000,False,LOW
8,PART_009,23,5,1.250000,False,LOW
9,PART_010,92,7,33.250000,False,LOW


In [41]:
total_parts = len(df)

low_stock_parts = (
    df["available_stock"] <= df["min_stock_level"]
).sum()

low_stock_rate = (
    low_stock_parts / total_parts * 100
)

reorder_suggestions = (
    df["reorder_required"] == True
).sum()

total_reorder_quantity = df["reorder_quantity"].sum()

estimated_reorder_cost = df["estimated_reorder_cost"].sum()

print("Total Parts:", total_parts)
print("Low Stock Parts:", low_stock_parts)
print("Low Stock Rate:", round(low_stock_rate, 2), "%")
print("Reorder Suggestions:", reorder_suggestions)
print("Total Reorder Quantity:", total_reorder_quantity)
print("Estimated Reorder Cost:", round(estimated_reorder_cost, 2))

Total Parts: 100
Low Stock Parts: 33
Low Stock Rate: 33.0 %
Reorder Suggestions: 34
Total Reorder Quantity: 3223
Estimated Reorder Cost: 8404770.11


In [42]:
reorder_df = df[df["reorder_required"] == True].copy()

reorder_df = reorder_df[
    [
        "part_id",
        "category",
        "available_stock",
        "min_stock_level",
        "max_stock_level",
        "lead_time_demand",
        "reorder_quantity",
        "estimated_reorder_cost",
        "reorder_reason",
        "urgency"
    ]
]

reorder_df.head(20)

,part_id,category,available_stock,min_stock_level,max_stock_level,lead_time_demand,reorder_quantity,estimated_reorder_cost,reorder_reason,urgency
0,PART_001,Suspension,0,21,50,12.857143,50,229165.00,Stock is at or below minimum level,HIGH
5,PART_006,Brake,11,24,97,18.500000,86,290449.52,Stock is at or below minimum level,HIGH
6,PART_007,Electrical,24,25,138,2.892857,114,383384.28,Stock is at or below minimum level,MEDIUM
10,PART_011,Suspension,4,22,113,16.250000,109,215421.06,Stock is at or below minimum level,HIGH
11,PART_012,Electrical,3,29,112,22.285714,109,529891.51,Stock is at or below minimum level,HIGH
14,PART_015,Suspension,0,7,142,29.750000,142,178398.86,Stock is at or below minimum level,HIGH
15,PART_016,Brake,0,12,116,11.142857,116,157148.68,Stock is at or below minimum level,HIGH
17,PART_018,Transmission,5,28,75,24.000000,70,250757.50,Stock is at or below minimum level,HIGH
20,PART_021,Brake,19,26,135,34.357143,116,126256.72,Stock is at or below minimum level,HIGH
21,PART_022,Transmission,0,27,106,11.142857,106,475859.44,Stock is at or below minimum level,HIGH
